In [1]:
import sys
import os
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql.column import Column
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [2]:
APP_NAME = "Discharge App - Gold"

In [3]:
read_csv = lambda path: spark.read.option("header", "true").option("sep", ";").csv(path)
DATA_PATH = os.getenv("DATA_PATH")

## Dimensions

In [4]:
dim_cause_of_death_df = read_csv(f"{DATA_PATH}/raw/dimensions/dim_cause_of_death.csv")\
    .select(
        F.col("id"),
        F.col("subcategory_code")
    )

dim_cause_of_death_df.show()

+---+----------------+
| id|subcategory_code|
+---+----------------+
|  1|            A000|
|  2|            A001|
|  3|            A009|
|  4|            A010|
|  5|            A011|
|  6|            A012|
|  7|            A013|
|  8|            A014|
|  9|            A020|
| 10|            A021|
| 11|            A022|
| 12|            A028|
| 13|            A029|
| 14|            A030|
| 15|            A031|
| 16|            A032|
| 17|            A033|
| 18|            A038|
| 19|            A039|
| 20|            A040|
+---+----------------+
only showing top 20 rows



In [5]:
dim_cause_of_death_df.filter((F.col("subcategory_code").isNull()) | (F.col("subcategory_code") == "")).show()

+---+----------------+
| id|subcategory_code|
+---+----------------+
+---+----------------+



In [7]:
cod_recod_df = read_csv(f"{DATA_PATH}/raw/dimensions/cause_of_death.recoded.lookup.csv")
cod_recod_df.show()

+-----+--------+
| code|new_code|
+-----+--------+
| A019|    A014|
| A089|    A085|
|A181+|    A181|
|  A35|    A35X|
|B023+|    B023|
|B572+|    B572|
|  B99|    B99X|
| C57X|    C579|
| C60X|    C609|
| C63X|    C639|
|D489+|    D489|
| E044|    E049|
|E123+|    E123|
|E133+|    E133|
|E142+|    E142|
|E143+|    E143|
| E14X|    E14X|
|H622*|    H622|
|  I44|    I447|
|  I64|    I64X|
+-----+--------+
only showing top 20 rows



In [20]:
cod_recod_df\
    .filter(F.col("code").rlike("^[a-zA-Z]{2}"))\
    .groupBy("code")\
    .agg(F.count("code").alias("count"))\
    .show()

+----+-----+
|code|count|
+----+-----+
+----+-----+



## Fact table

In [35]:
discharges_df = spark\
    .read\
    .parquet(f"{DATA_PATH}/silver/cleaned_discharges")

discharges_df.show()

[Stage 60:===========================================>              (3 + 1) / 4]

+----+------+---+----------------+----+------------------+----------------+----------------+
|year|gender|age|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2|
+----+------+---+----------------+----+------------------+----------------+----------------+
|2001|     2| 37|               1|1101|                 1|            O039|             N/E|
|2001|     1| 55|               1|5108|                 7|            M139|             N/E|
|2001|     1| 33|               1|1101|                 1|            C629|             N/E|
|2001|     2| 30|               1|5108|                 7|            O809|             N/E|
|2001|     1|  6|               1|1101|                 1|            A09X|             N/E|
|2001|     2| 23|               1|5108|                 7|            O809|             N/E|
|2001|     2| 47|               1|1101|                 1|            K810|             N/E|
|2001|     2| 17|               1|5108|                 7|            

In [36]:
discharges_df\
    .groupBy("year")\
    .agg(F.count("year").alias("count"))\
    .sort(F.col("year").asc())\
    .show()

+----+-------+
|year|  count|
+----+-------+
|2001|1506890|
|2002|1549893|
|2003|1565913|
|2004|1592990|
|2005|1627743|
|2006|1637916|
|2007|1632887|
|2008|1608540|
|2009|1682056|
|2010|1623875|
|2011|1648687|
|2012|1659654|
|2013|1676932|
|2014|1660150|
|2015|1671091|
|2016|1637265|
|2017|1637150|
|2018|1669602|
+----+-------+



In [37]:
discharges_df\
    .filter(F.col("cause_of_death_2").rlike("^[a-zA-Z]{2}"))\
    .groupBy("year")\
    .agg(F.count("year").alias("count"))\
    .show()

[Stage 64:==================================================>       (7 + 1) / 8]

+----+-------+
|year|  count|
+----+-------+
|2003|      2|
|2013|      2|
|2004|      3|
|2012|1504065|
|2005|1474561|
|2011|1495066|
|2002|      1|
+----+-------+



## Pipeline

In [38]:
step_1_df = discharges_df\
    .alias("e")\
    .join(
        F.broadcast(cod_recod_df.alias("cod_1")),
        F.translate(F.col("e.cause_of_death_1"), ",.", "") == F.col("cod_1.code"),
        how="left"
    )

step_1_df.show()

[Stage 69:===========================================>              (3 + 1) / 4]

+----+------+---+----------------+----+------------------+----------------+----------------+----+--------+
|year|gender|age|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2|code|new_code|
+----+------+---+----------------+----+------------------+----------------+----------------+----+--------+
|2001|     2| 37|               1|1101|                 1|            O039|             N/E|null|    null|
|2001|     1| 55|               1|5108|                 7|            M139|             N/E|null|    null|
|2001|     1| 33|               1|1101|                 1|            C629|             N/E|null|    null|
|2001|     2| 30|               1|5108|                 7|            O809|             N/E|null|    null|
|2001|     1|  6|               1|1101|                 1|            A09X|             N/E|null|    null|
|2001|     2| 23|               1|5108|                 7|            O809|             N/E|null|    null|
|2001|     2| 47|               1|110

In [15]:
step_1_df\
    .filter(F.col("cause_of_death_2").rlike("^[a-zA-Z]{2}"))\
    .groupBy("year")\
    .agg(F.count("year").alias("count"))\
    .show()

[Stage 17:==================================================>       (7 + 1) / 8]

+----+-------+
|year|  count|
+----+-------+
|2003|      2|
|2013|      2|
|2004|      3|
|2012|1504065|
|2005|1474561|
|2011|1495066|
|2002|      1|
+----+-------+



In [39]:
step_2_df = discharges_df\
    .alias("e")\
    .join(
        F.broadcast(cod_recod_df.alias("cod_1")),
        F.translate(F.col("e.cause_of_death_1"), ",.", "") == F.col("cod_1.code"),
        how="left"
    )\
    .join(
        F.broadcast(cod_recod_df.alias("cod_2")),
        F.translate(F.col("e.cause_of_death_2"), ",.", "") == F.col("cod_2.code"),
        how="left"
    )

step_2_df.show()

[Stage 72:===========================================>              (3 + 1) / 4]

+----+------+---+----------------+----+------------------+----------------+----------------+----+--------+----+--------+
|year|gender|age|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2|code|new_code|code|new_code|
+----+------+---+----------------+----+------------------+----------------+----------------+----+--------+----+--------+
|2001|     2| 37|               1|1101|                 1|            O039|             N/E|null|    null|null|    null|
|2001|     1| 55|               1|5108|                 7|            M139|             N/E|null|    null|null|    null|
|2001|     1| 33|               1|1101|                 1|            C629|             N/E|null|    null|null|    null|
|2001|     2| 30|               1|5108|                 7|            O809|             N/E|null|    null|null|    null|
|2001|     1|  6|               1|1101|                 1|            A09X|             N/E|null|    null|null|    null|
|2001|     2| 23|               

In [17]:
step_2_df\
    .filter(F.col("cause_of_death_2").rlike("^[a-zA-Z]{2}"))\
    .groupBy("year")\
    .agg(F.count("year").alias("count"))\
    .show()

[Stage 24:==================================================>       (7 + 1) / 8]

+----+-------+
|year|  count|
+----+-------+
|2003|      2|
|2013|      2|
|2004|      3|
|2012|1504065|
|2005|1474561|
|2011|1495066|
|2002|      1|
+----+-------+



In [40]:
step_3_df = discharges_df\
    .alias("e")\
    .join(
        F.broadcast(cod_recod_df.alias("cod_1")),
        F.translate(F.col("e.cause_of_death_1"), ",.", "") == F.col("cod_1.code"),
        how="left"
    )\
    .join(
        F.broadcast(cod_recod_df.alias("cod_2")),
        F.translate(F.col("e.cause_of_death_2"), ",.", "") == F.col("cod_2.code"),
        how="left"
    )\
    .select(
        "*",
        F.when(
            F.coalesce(
                F.translate(F.col("cod_1.new_code"), ",.", ""),
                F.translate(F.col("e.cause_of_death_1"), ",.", "")
            ).rlike("^[VWXY]"), 
            F.lit("N/E")
        ).otherwise(
            F.coalesce(
                F.translate(F.col("cod_1.new_code"), ",.", ""),
                F.translate(F.col("e.cause_of_death_1"), ",.", "")
            )
        ).alias("cause_of_death_1_final"),
        F.coalesce(
            F.translate(F.col("cod_2.new_code"), ",.", ""),
            F.translate(F.col("e.cause_of_death_2"), ",.", "")
        ).alias("cause_of_death_2_final")
    )

step_3_df.show()

[Stage 75:===========================================>              (3 + 1) / 4]

+----+------+---+----------------+----+------------------+----------------+----------------+----+--------+----+--------+----------------------+----------------------+
|year|gender|age|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2|code|new_code|code|new_code|cause_of_death_1_final|cause_of_death_2_final|
+----+------+---+----------------+----+------------------+----------------+----------------+----+--------+----+--------+----------------------+----------------------+
|2001|     2| 37|               1|1101|                 1|            O039|             N/E|null|    null|null|    null|                  O039|                   N/E|
|2001|     1| 55|               1|5108|                 7|            M139|             N/E|null|    null|null|    null|                  M139|                   N/E|
|2001|     1| 33|               1|1101|                 1|            C629|             N/E|null|    null|null|    null|                  C629|                   N/E

In [19]:
step_3_df\
    .filter(F.col("cause_of_death_2").rlike("^[a-zA-Z]{2}"))\
    .groupBy("year")\
    .agg(F.count("year").alias("count"))\
    .show()

[Stage 31:==================================================>       (7 + 1) / 8]

+----+-------+
|year|  count|
+----+-------+
|2003|      2|
|2013|      2|
|2004|      3|
|2012|1504065|
|2005|1474561|
|2011|1495066|
|2002|      1|
+----+-------+



In [46]:
step_4_df = discharges_df\
    .alias("e")\
    .join(
        F.broadcast(cod_recod_df.alias("cod_1")),
        F.translate(F.col("e.cause_of_death_1"), ",.", "") == F.col("cod_1.code"),
        how="left"
    )\
    .join(
        F.broadcast(cod_recod_df.alias("cod_2")),
        F.translate(F.col("e.cause_of_death_2"), ",.", "") == F.col("cod_2.code"),
        how="left"
    )\
    .select(
        "*",
        F.when(
            F.coalesce(
                F.translate(F.col("cod_1.new_code"), ",.", ""),
                F.translate(F.col("e.cause_of_death_1"), ",.", "")
            ).rlike("^[VWXY]"), 
            F.lit("N/E")
        ).otherwise(
            F.coalesce(
                F.translate(F.col("cod_1.new_code"), ",.", ""),
                F.translate(F.col("e.cause_of_death_1"), ",.", "")
            )
        ).alias("cause_of_death_1_final"),
        F.coalesce(
            F.translate(F.col("cod_2.new_code"), ",.", ""),
            F.translate(F.col("e.cause_of_death_2"), ",.", "")
        ).alias("cause_of_death_2_final")
    )\
    .alias("e")\
    .join(
        F.broadcast(dim_cause_of_death_df.alias("c1")),
        F.col("e.cause_of_death_1_final") == F.col("c1.subcategory_code"),
        how="left"
    )\
    .join(
        F.broadcast(dim_cause_of_death_df.alias("c2")),
        F.col("e.cause_of_death_2_final") == F.col("c2.subcategory_code"),
        how="left"
    )\
    .select(
        F.col("e.health_insurance"),
        F.col("e.year"),
        F.col("e.gender"),
        F.col("e.age"),
        F.col("e.city"),
        F.col("e.healthcare_service"),
        F.col("cause_of_death_1"),
        F.col("cause_of_death_2"),
        F.col("e.cause_of_death_1_final"),
        F.col("e.cause_of_death_2_final"),
        F.coalesce(F.col("c1.id"), F.lit(999999)).alias("main_cause_of_death_id"),
        F.coalesce(F.col("c2.id"), F.lit(999999)).alias("external_cause_of_death_id")
    )

step_4_df.show()

[Stage 95:===========================================>              (3 + 1) / 4]

+----------------+----+------+---+----+------------------+----------------+----------------+----------------------+----------------------+----------------------+--------------------------+
|health_insurance|year|gender|age|city|healthcare_service|cause_of_death_1|cause_of_death_2|cause_of_death_1_final|cause_of_death_2_final|main_cause_of_death_id|external_cause_of_death_id|
+----------------+----+------+---+----+------------------+----------------+----------------+----------------------+----------------------+----------------------+--------------------------+
|               1|2001|     2| 37|1101|                 1|            O039|             N/E|                  O039|                   N/E|                  5635|                    999999|
|               1|2001|     1| 55|5108|                 7|            M139|             N/E|                  M139|                   N/E|                  4699|                    999999|
|               1|2001|     1| 33|1101|                

In [48]:
step_4_df\
    .filter(
        (F.col("cause_of_death_2") == "N/E") |
        (F.col("cause_of_death_2") == "NULL") |
        (F.col("cause_of_death_2").isNull())
    )\
    .groupBy("year")\
    .count()\
    .sort(F.col("year").asc())\
    .show()

[Stage 103:=================================================>       (7 + 1) / 8]

+----+-------+
|year|  count|
+----+-------+
|2001|1398297|
|2002|1415437|
|2003|1450056|
|2004|1455212|
|2005|1474561|
|2006|1479707|
|2007|1480092|
|2008|1448875|
|2009|1525410|
|2010|1471099|
|2011|1495269|
|2012|1504065|
|2013|1523005|
|2014|1508883|
|2015|1517576|
|2016|1485188|
|2017|1482586|
|2018|1515073|
+----+-------+



## Letters pattern

In [28]:
filtered_df = discharges_df\
    .withColumn("second_letter", F.substring(F.col("cause_of_death_2"), 2, 1))

filtered_df.show()

[Stage 44:===========================================>              (3 + 1) / 4]

+----+------+---+----------------+----+------------------+----------------+----------------+-------------+
|year|gender|age|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2|second_letter|
+----+------+---+----------------+----+------------------+----------------+----------------+-------------+
|2001|     2| 37|               1|1101|                 1|            O039|             N/E|            /|
|2001|     1| 55|               1|5108|                 7|            M139|             N/E|            /|
|2001|     1| 33|               1|1101|                 1|            C629|             N/E|            /|
|2001|     2| 30|               1|5108|                 7|            O809|             N/E|            /|
|2001|     1|  6|               1|1101|                 1|            A09X|             N/E|            /|
|2001|     2| 23|               1|5108|                 7|            O809|             N/E|            /|
|2001|     2| 47|               1|110

In [34]:
filtered_df\
    .filter(
        (F.col("cause_of_death_2").rlike("^[a-zA-Z]{1}")) &
        (F.col("cause_of_death_2").isNotNull()) &
        (F.col("cause_of_death_2") != "NULL") &
        (F.col("cause_of_death_2") != "N/E") & 
        (F.col("cause_of_death_2").startswith("V"))
    )\
    .show()

[Stage 57:===========================================>              (3 + 1) / 4]

+----+------+---+----------------+----+------------------+----------------+----------------+-------------+
|year|gender|age|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2|second_letter|
+----+------+---+----------------+----+------------------+----------------+----------------+-------------+
|2001|     1|  0|               4|1101|                 1|            S411|            V439|            4|
|2001|     1| 15|               1|1101|                 1|            T149|            V184|            1|
|2001|     2| 61|               2|1101|                 1|            S021|            V439|            4|
|2001|     1| 53|               3|1101|                 1|            T149|            V439|            4|
|2001|     2|  1|               4|1101|                 1|            S069|            V439|            4|
|2001|     2| 11|               4|1101|                 1|            S068|            V439|            4|
|2001|     1| 23|               4|110